# AU Case Study - Region & Feature Assembly Overview (Read-Only)

> Data sources: `008_build_regions.py` (region assembly) / `009_build_grid.py` (agent grid) /
> `010_build_features.py` (feature extraction); specification = `docs/B0_design_decisions.md`.
> This notebook is **read-only display only, and writes no artifacts**; all numbers come
> from saved files and registries on disk (nothing is hand-typed).
>
> **Count correction**: the original design note listed SA3=35 / SA4=13, following the
> earlier full 216-point layer convention. Applying the universe rule (an SA3 must contain
> at least one **usable station**; FY2009 usable count = 143), SA3 11502
> "Dural - Wisemans Ferry" (whose only station, Galston, was a FY2009 placeholder that only
> went into service in 2025) and its SA4 "Sydney - Baulkham Hills and Hawkesbury" drop out --
> **actual count = 34 / 12**. See the deviation record in
> `docs/b1b2/b1_regions.json.deviation_from_b0`.

In [ ]:
%matplotlib inline
# Environment and artifact paths (read-only)
import json
import pickle
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

AU_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED = AU_DIR / "data" / "processed"
GRID_DIR = PROCESSED / "grid"
EXTRACTED = PROCESSED / "features" / "extracted"
ASSEMBLED = PROCESSED / "features" / "assembled"
DOCS_B1B2 = AU_DIR / "docs" / "b1b2"

b1 = json.loads((DOCS_B1B2 / "b1_regions.json").read_text(encoding="utf-8"))
b2 = json.loads((DOCS_B1B2 / "b2_features.json").read_text(encoding="utf-8"))
print("Region counts:", b1["counts"])
print("Deviation record:", b1["deviation_from_b0"]["actual"], "(original spec",
      b1["deviation_from_b0"]["b0_pinned"], ")")

## 1. Region Assembly

34 source SA3 units (colored by parent SA4) and 143 usable stations; regional demand =
sum of usable-station `peak_mw` within each region (primary convention).

In [ ]:
# Region map: SA3 colored by SA4 + usable stations
sa3 = gpd.read_file(PROCESSED / "regions_sa3.gpkg", layer="regions_sa3")
sa4 = gpd.read_file(PROCESSED / "regions_sa4.gpkg", layer="regions_sa4")
st = pd.read_csv(PROCESSED / "station_table_fy2009.csv", encoding="utf-8-sig")
usable = st[st["status"].isin(["matched", "matched_osm"])]
pts = gpd.GeoDataFrame(usable, geometry=gpd.points_from_xy(
    usable["lon_wgs84"], usable["lat_wgs84"]), crs="EPSG:4326")

fig, ax = plt.subplots(figsize=(10, 11))
sa3.plot(ax=ax, column="sa4_name", cmap="tab20", edgecolor="white",
         linewidth=0.8, legend=True,
         legend_kwds={"loc": "upper left", "fontsize": 8, "title": "Evaluation region (SA4)"})
sa4.boundary.plot(ax=ax, color="black", linewidth=1.2)
pts.plot(ax=ax, color="crimson", markersize=8, zorder=5, label="Usable stations (143)")
ax.set_title(f"AU footprint: source SA3 x {len(sa3)} | evaluation SA4 x {len(sa4)} | usable stations x {len(pts)}")
ax.set_axis_off()
plt.show()

In [ ]:
# SA3 macro attribute table (key columns from region_attributes.csv)
attrs = pd.read_csv(PROCESSED / "region_attributes.csv", encoding="utf-8-sig",
                    dtype={"sa3_code": str, "sa4_code": str})
show_cols = ["sa3_code", "sa3_name", "sa4_name", "n_stations_usable",
             "demand_peak_mw", "population_erp_2009", "income_total_aud_2009",
             "residential_percent", "industrial_percent", "commercial_percent",
             "agricultural_percent", "others_percent"]
with pd.option_context("display.float_format", "{:,.4f}".format):
    display(attrs[show_cols].sort_values("demand_peak_mw", ascending=False)
            .reset_index(drop=True))
print(f"Demand conservation check: Sum SA3 demand_peak_mw = {attrs['demand_peak_mw'].sum():,.3f} MW "
      f"= Sum usable-station peak_mw = {usable['peak_mw'].sum():,.3f} MW")

### Border SA3 Visual Check

Usable-station Voronoi coverage vs. SA3 area -- border SA3 units with low coverage flag
"demand potentially spread outside the Ausgrid service area" (overlap with the Endeavour
service area), disclosed for the paper's limitations section.

In [ ]:
from IPython.display import Image, display as idisplay
idisplay(Image(str(DOCS_B1B2 / "border_sa3_voronoi_check.png"), width=760))
border = pd.read_csv(DOCS_B1B2 / "border_sa3_check.csv", encoding="utf-8-sig")
print("5 SA3 units with the lowest own_voronoi_share (leakage risk flag):")
display(border.nsmallest(5, "own_voronoi_share").reset_index(drop=True))

## 2. Agent Grid

Uses the exact same functions as the UK case study (`calculate_step_size` /
`generate_base_grid`) with the same parameters (target=50000/SA4, min_step=10 m,
EPSG:3857); step size is endogenous to region area -- the span from 40 m in inner
Sydney to 770 m in Hunter Valley is the natural result of this design.

In [ ]:
step_table = pd.read_csv(GRID_DIR / "grid_step_size_table.csv", encoding="utf-8-sig")
display(step_table)
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
step_sorted = step_table.sort_values("step_size_m")
axes[0].barh(step_sorted["loc_key"], step_sorted["step_size_m"], color="steelblue")
axes[0].set_xlabel("Step size (m, nominal EPSG:3857)")
axes[0].set_title("Per-region grid step size")
axes[1].barh(step_sorted["loc_key"], step_sorted["n_points"], color="darkseagreen")
axes[1].axvline(50000, color="crimson", linestyle="--", label="target=50000")
axes[1].set_xlabel("Grid point count")
axes[1].set_title("Per-region grid point count")
axes[1].legend()
axes[1].set_yticklabels([])
plt.tight_layout()
plt.show()
print(f"Total grid points: {step_table['n_points'].sum():,}")

## 3. Feature Distributions

### 3.1 Five Land-Use Category Shares (CLUM 2015-03 reclassification)

Mapping table = ALUM v7 secondary codes plus tertiary exceptions; the hard-fail
assertion for unknown codes has passed (the per-region window code-list registry
is recorded in `b2_features.json`).

In [ ]:
# lu_* stacked bar chart of region-mean shares (from the b2 registry's footprint_group_share)
groups = ["residential", "commercial", "industrial", "agricultural", "others"]
colors = {"residential": "#d95f02", "commercial": "#7570b3", "industrial": "#666666",
          "agricultural": "#66a61e", "others": "#a6cee3"}
rows = {loc: e["landuse"]["footprint_group_share"] for loc, e in b2["regions"].items()}
lu_share = pd.DataFrame(rows).T.loc[sorted(rows)][groups]
ax = lu_share.plot(kind="barh", stacked=True, figsize=(10, 5),
                   color=[colors[g] for g in groups])
ax.set_xlabel("Grid-point mean share within region")
ax.set_title("lu_* five-category shares - region mean (CLUM 2015-03 reclassification)")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
plt.tight_layout(); plt.show()
print("Note: CLUM class 551 (commercial services) is nearly absent within the footprint (only 246 raster pixels) --")
print("lu_commercial_prop is near zero domain-wide; urban commercial land is mostly classified by CLUM under")
print("code 540 (residential) or the 55x service codes -- this is a property of the source data rather than a")
print("pipeline error (to be disclosed in the paper's methods section as one substantive difference from the UK OSM convention).")

### 3.2 NTL (DMSP-OLS F162008+F162009 median composite)

DMSP DN value range 0-63; urban-core saturation is a known residual risk --
**every grid point in Sydney_Inner_West = 63 (variance is exactly 0, flagged
`fully_saturated`)**, so the multiplicative NTL factor degenerates to a constant
in that region -- which falls squarely within the disclosure scope of the
"mechanism direction" comparison framework.

In [ ]:
ntl_stats = pd.DataFrame({loc: e["ntl_stats"] for loc, e in b2["regions"].items()}).T
ntl_stats = ntl_stats.loc[sorted(ntl_stats.index)]
display(ntl_stats)

all_ntl = np.concatenate([
    np.load(EXTRACTED / f"{loc}_ntl.npz", allow_pickle=True)["data"][:, 0]
    for loc in b2["regions"]])
fig, ax = plt.subplots(figsize=(9, 3.5))
ax.hist(all_ntl, bins=64, range=(0, 64), color="goldenrod", log=True)
ax.axvline(63, color="crimson", linestyle="--", label="DN saturation ceiling = 63")
ax.set_xlabel("stable_lights DN"); ax.set_ylabel("Grid point count (log)")
ax.set_title(f"Full-footprint NTL distribution ({len(all_ntl):,} points; variance = {all_ntl.var():.1f})")
ax.legend(); plt.tight_layout(); plt.show()

### 3.3 Proximity (UK formula, TARGET_CRS=EPSG:7856) and wc_* Diagnostic Columns

In [ ]:
prox = pd.DataFrame({loc: e["proximity_stats"] for loc, e in b2["regions"].items()}).T
prox = prox.loc[sorted(prox.index)]
display(prox)
print(f"Total station count across regions = {int(prox['n_stations'].sum())} (= full usable-station set of 143, region-slice convention)")

wc = pd.DataFrame({loc: {**e["worldcover"],
                          "n_marine_lu_others": e["landuse"]["n_marine_nodata_others"]}
                   for loc, e in b2["regions"].items()}).T
wc = wc.loc[sorted(wc.index)]
display(wc)
print("WorldCover coverage gap: no downloaded tile covers the footprint's western edge 149.79-150.0 deg E")
print("(tiles were selected by station bounding box); only 908 Hunter Valley points are affected;")
print("wc_* are diagnostic columns and are not used in the paper's results.")
print("n_marine_lu_others = number of points where CLUM ocean nodata falls back to the 'others' category")
print("(following the same convention as the UK case study's default_category).")

### 3.4 Assembled Artifact Schema (aligned with the UK grid gdf)

In [ ]:
with open(ASSEMBLED / "Sydney_Ryde_grid_points.pickle", "rb") as fh:
    gdf, step = pickle.load(fh)
print(f"Example Sydney_Ryde: {len(gdf):,} points, step={step:.0f} m, CRS={gdf.crs}")
print("Columns (aligned with UK [geometry,index_region,ITL3,ITL2]+lu_5+wc_3 -> AU SA3/SA4):")
print(list(gdf.columns))
display(gdf.drop(columns="geometry").head(3))
schema = json.loads((ASSEMBLED / "feature_schema.json").read_text(encoding="utf-8"))
print("feature_schema numerical_col_names:", schema["numerical_col_names"])

---
*Generated 2026-07-14. Verified by `tests/test_b1b2.py` (19 checks), `docs/b1b2/b1_regions.json` / `b2_features.json`.*